# Lab Day 19 — GraphRAG với US EV Industry Corpus
**Sinh viên:** Lương Quốc Dũng — 2A202600601  
**Corpus:** 70 văn bản web thô về ngành xe điện (EV) Mỹ (giáo viên cấp)  
**Stack:** NetworkX + gpt-5.4-nano (OpenAI) + ChromaDB + sentence-transformers

In [ ]:
import sys, os
sys.path.insert(0, 'src')
print('Working dir:', os.getcwd())

## 1. Environment Setup

In [ ]:
from config import make_openai_client, OPENAI_MODEL, DATASET_DIR
import glob
n_docs = len(glob.glob(os.path.join(DATASET_DIR, '*.txt')))
c = make_openai_client()
r = c.chat.completions.create(model=OPENAI_MODEL,
    messages=[{'role':'user','content':'Say READY'}], max_completion_tokens=10)
print(f'Model: {OPENAI_MODEL}')
print(f'Connection: {r.choices[0].message.content}')
print(f'Corpus: {n_docs} text documents')

## 2. Bước 1 — Indexing: Trích Triples từ VĂN BẢN THÔ
LLM đọc từng doc, làm sạch boilerplate, trích quan hệ entity-to-entity.

In [ ]:
from extract_text import parse_doc, build_triples_from_text

# Xem thử một doc đã parse
sample = parse_doc(os.path.join(DATASET_DIR, 'doc_2.txt'))
print('Title :', sample['title'][:80])
print('Query :', sample['query'])
print('Content (200 chars):', sample['content'][:200])

In [ ]:
import json
# Trích triples (dùng cache nếu đã chạy — tránh tốn token). Bỏ comment để chạy lại từ đầu:
# payload = build_triples_from_text(use_cache=True)
payload = json.load(open('outputs/triples.json', encoding='utf-8'))
print('Docs    :', payload['counts']['docs'])
print('Triples :', payload['counts']['triples'])
print('Entities:', payload['counts']['entities'])
print('Tokens  :', payload['usage']['total_tokens'], f"(~${payload['usage']['est_cost_usd']:.4f})")

In [ ]:
# Mẫu triples trích được
for t in payload['triples'][:15]:
    print(f"  ({t['subject']}) --[{t['relation']}]--> ({t['object']})")

## 3. Bước 2 — Construction: Knowledge Graph (NetworkX + dedup)

In [ ]:
from graph_build import build_graph, save_graph, draw_graph, print_stats
G = build_graph()
print_stats(G)
save_graph(G)

In [ ]:
# Vẽ Knowledge Graph (Deliverable #2) — top 60 node bậc cao nhất
from IPython.display import Image
img_path = draw_graph(G)
print(f'Graph saved: {img_path}')
Image(img_path, width=1000)

In [ ]:
# Demo 2-hop traversal quanh node Tesla
import networkx as nx
ego = nx.ego_graph(G, 'Tesla', radius=2, undirected=True)
print(f'2-hop quanh Tesla: {ego.number_of_nodes()} nodes, {ego.number_of_edges()} edges')
print('Quan hệ trực tiếp của Tesla:')
for u, v, d in G.out_edges('Tesla', data=True):
    print(f"  Tesla --[{d['rel']}]--> {v}")

## 4. Bước 3 — Querying: Flat RAG vs GraphRAG

In [ ]:
from flat_rag import FlatRAG
from graph_rag import GraphRAG

flat = FlatRAG()
print(f'Flat RAG indexed: {flat.index()} chunks')
grag = GraphRAG(G=G)
print('GraphRAG ready')

In [ ]:
# Câu hỏi tiêu biểu — Flat RAG hay ảo giác, GraphRAG trả lời đúng
for q in ['What does Nikola Corporation supply?',
          'Which company partnered with Honda?',
          'Who is the CEO of the company that produces the Model Y?']:
    r1 = flat.query(q); r2 = grag.query(q)
    print(f'\nQ: {q}')
    print(f'Flat RAG : {r1["answer"][:160]}')
    print(f'GraphRAG : {r2["answer"][:160]}')
    print('-'*70)

## 5. Bước 4 — Evaluation: Benchmark 20 câu hỏi

In [ ]:
from evaluate import run_benchmark
results = run_benchmark()

In [ ]:
import pandas as pd
df = pd.read_csv('outputs/benchmark_results.csv')
flat_acc  = (df['flat_correct']  == 'CORRECT').sum()
graph_acc = (df['graph_correct'] == 'CORRECT').sum()
halluc    = (df['hallucination_caught'] == 'YES').sum()
print(f'Flat RAG  : {flat_acc}/20 ({flat_acc/20*100:.0f}%)')
print(f'GraphRAG  : {graph_acc}/20 ({graph_acc/20*100:.0f}%)')
print(f'Hallucination caught: {halluc}')
df[['id','hop','question','flat_correct','graph_correct','hallucination_caught']]

In [ ]:
with open('outputs/cost_report.md', encoding='utf-8') as f:
    print(f.read())

## 6. Kết luận

| | Flat RAG | GraphRAG |
|---|---|---|
| Accuracy | 55% | **80%** |
| Hallucination caught | — | **7** |
| Total cost | — | ~$0.034 (180k tokens) |

Trên corpus văn bản thô EV, GraphRAG vượt Flat RAG **25 điểm %** nhờ duyệt quan hệ trực tiếp
trong đồ thị tri thức, thay vì retrieval chunk rời rạc dễ ảo giác.